### Let's Learn About Embeddings using Ollama and HuggingFace Embeddings 

In [5]:
# Let's See About Ollama 
import ollama
from langchain_community.embeddings import OllamaEmbeddings

response = ollama.embed(
    model='nomic-embed-text-v2-moe',
    input='The sky is blue because of Rayleigh scattering',
)
print(response.embeddings)

[[-0.034061864, 0.030212874, -0.07470542, -0.051189218, -0.035281647, -0.03898889, -0.019954575, 0.023844598, 0.03919646, 0.057879116, 0.066912755, -0.08666106, 0.015222331, 0.04458797, 0.020813726, -0.001627181, -0.046529975, -0.039929956, 0.0028523311, -0.04228345, -0.018951938, 0.004555421, -0.043910004, -0.07478413, 0.059202507, 0.024800355, -0.025439031, 0.019389084, -0.0020103445, 0.017512223, 0.014552638, 0.018607777, 0.028801247, 0.040744487, 0.025711177, 0.028061608, -0.012766156, -0.0049105063, -0.017993253, 0.013539403, 0.014393223, 0.026533261, -0.013261679, 0.040988863, -0.016966967, 0.013954349, 0.015748858, 0.033435732, -0.029394655, -0.018972842, -0.017969264, -0.09548355, -0.03732311, -0.06176345, -0.0078112083, -0.045794636, -0.01126429, 0.002024605, 0.09370017, -0.0362695, -0.03919912, 0.0134476805, 0.06889971, 0.042005368, -0.012101311, -0.04600938, -0.028908506, 0.034529787, -0.081568636, 0.020944111, -0.0016719024, 0.004164096, -0.003997077, 0.02197839, -0.0443385

In [6]:
ollama_embeddings = OllamaEmbeddings(
    model = "nomic-embed-text-v2-moe"
)

C:\Users\mahen\AppData\Local\Temp\ipykernel_10872\93759304.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  ollama_embeddings = OllamaEmbeddings(


In [7]:
# This is our embedding model which we have setup 
ollama_embeddings

OllamaEmbeddings(base_url='http://localhost:11434', model='nomic-embed-text-v2-moe', embed_instruction='passage: ', query_instruction='query: ', mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None, show_progress=False, headers=None, model_kwargs=None)

In [9]:
# Let's perform embedding as well as vector store and vector search

# 1. Read file 
from pathlib import Path
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Read the pdf 
BASE_DIR = Path.cwd().resolve().parent
pdf_file_path = BASE_DIR / "config" / "data" / "raw" / "demo.txt"

loader = PyMuPDFLoader(str(pdf_file_path))
text_documents_from_pdf = loader.load()

# 2. Perform Chunking 
pdf_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=100, chunk_overlap=50, separators=[
        "\n# ",
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " "
    ])

# chunking 
splitted_documents = pdf_splitter.split_documents(text_documents_from_pdf)



# 3. Let's perform the embedding and store it inside the chroma db
db = Chroma.from_documents(splitted_documents, embedding=ollama_embeddings)
db


### Huggingface Embeddings 

In [ ]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Read the pdf 
BASE_DIR = Path.cwd().resolve().parent
pdf_file_path = BASE_DIR / "config" / "data" / "processed" / "AttentionYou.pdf"

loader = PyMuPDFLoader(str(pdf_file_path))
text_documents_from_pdf = loader.load()

# 2. Perform Chunking 
pdf_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=800, chunk_overlap=150, separators=[
        "\n# ",
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        ". ",
        "? ",
        "! ",
        " "
    ])

# chunking 
splitted_documents = pdf_splitter.split_documents(text_documents_from_pdf)

# 1. Load environment variables from the .env file
load_dotenv()

# Verify the token is loaded into the environment (Optional check)
if not os.getenv("HF_TOKEN"):
    raise ValueError("HF_TOKEN not found! Please check your .env file.")

# 2. Initialize the blazing-fast embedding model
# The model will download locally on its first run and cache itself.
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 3. Test the embedding model to see how fast it runs
db2 = Chroma.from_documents(splitted_documents, embedding_model, collection_name="attention_pdf_hf")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10620.84it/s]


In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder
embeded_model = SentenceTransformer(
    model_name_or_path="BAAI/bge-reranker-base"
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1981.03it/s]
[transformers] XLMRobertaModel LOAD REPORT from: BAAI/bge-reranker-base
Key                        | Status     | 
---------------------------+------------+-
classifier.out_proj.bias   | UNEXPECTED | 
classifier.dense.weight    | UNEXPECTED | 
classifier.dense.bias      | UNEXPECTED | 
classifier.out_proj.weight | UNEXPECTED | 
pooler.dense.bias          | MISSING    | 
pooler.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [2]:
from sentence_transformers import SentenceTransformer, CrossEncoder
embeded_model = SentenceTransformer(
    model_name_or_path="BAAI/bge-small-en-v1.5"
)

d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahen\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00

In [ ]:
document_chunks = [
    "Attention mechanisms allow deep learning architectures to dynamically compute token weight priorities.",
    "Bypassing external microservices via native local pipelines drops data transmission overhead to zero.",
    "Vector indices serve as the fundamental indexing system for enterprise Retrieval-Augmented Generation."
]
vector = embeded_model.encode(document_chunks, batch_size=32, normalize_embeddings=True)

In [5]:
vector.tolist()

[-0.06177574396133423,
 -0.03407224267721176,
 0.03345828130841255,
 -0.029769139364361763,
 -0.054700687527656555,
 -0.017571421340107918,
 0.10383977741003036,
 0.002795728389173746,
 0.03629940748214722,
 -0.020221617072820663,
 0.0013431397965177894,
 -0.05932598188519478,
 0.044427622109651566,
 -0.010517816059291363,
 0.06615370512008667,
 0.013417340815067291,
 0.045531224459409714,
 -0.03231222182512283,
 -0.1203741654753685,
 -0.007887021638453007,
 -0.010353665798902512,
 0.01908610202372074,
 -0.11531275510787964,
 0.010917636565864086,
 0.022418836131691933,
 0.016925420612096786,
 0.05682212486863136,
 -0.008794300258159637,
 0.0006925184279680252,
 -0.06084849312901497,
 -0.027967585250735283,
 0.03918863832950592,
 0.01823207549750805,
 0.028975967317819595,
 -0.03881341591477394,
 0.008646072819828987,
 -0.028987623751163483,
 0.04090219363570213,
 0.00946446880698204,
 -0.03590230643749237,
 -0.00033745073596946895,
 -0.07993002980947495,
 0.060763727873563766,
 -0.072

In [15]:
db2

In [16]:
# Let's search the result from the Chroma DB 
query = "what is scaler dot product attention ?"
retrieve_result = db2.similarity_search(query)
retrieve_result

[Document(metadata={'file_path': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'title': 'Attention is All you Need', 'creationDate': '', 'source': 'D:\\AI-Restart-2026\\Complete GEN AI Journey - Notes\\DataIngestion_Pipeline\\config\\data\\processed\\AttentionYou.pdf', 'total_pages': 11, 'creationdate': '', 'creator': '', 'producer': 'PyPDF2', 'trapped': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'moddate': '2018-02-12T21:22:10-08:00', 'modDate': "D:20180212212210-08'00'", 'keywords': '', 'page': 2, 'format': 'PDF 1.3'}, page_content='of the values, where the weight assigned to each value is computed by a compatibility function of the\nquery with the corresponding key.\n3.2.1\nScaled Dot-Product Attention\nWe call our particular attention "Scaled Dot

In [10]:

from pathlib import Path
BASE_DIR = Path.cwd()
pdf_upload_directory = BASE_DIR / "config" / "data" / "uploads" / "Neural-Network(Basics).pdf"
pdf_upload_directory.name

'Neural-Network(Basics).pdf'

In [1]:
chunks_data = [{'id': 'Neural-Network(Basics).pdf_chunk16', 'text': 'Activation Functions (cont’d)\n\n● Sigmoid\n\n ○ continuously differentiable\n ○ ranges from 0-1\n ○ not symmetric around the origin\n● Tanh\n ○\n ○\n ○ scaled version of the sigmoid\n ○ symmetric around the origin\n ○ vanishing gradient\n\n 17', 'metadata': {'source': 'Neural-Network(Basics).pdf'}, 'distance': 0.37272316217422485, 'rerank_score': 0.0008774126181378961}, {'id': 'Neural-Network(Basics).pdf_chunk27', 'text': 'Optimization (cont’d)\n\nFind w which minimizes the chosen\nerror function E(w)\n\n● wA : a local minimum\n● wB : a global minimum\n● At point wC local gradient is given\n by vector ΔE(w)\n● It points in direction of greatest\n rate of increase of E(w)\n● Negative gradient points to rate\n of greatest decrease 28', 'metadata': {'source': 'Neural-Network(Basics).pdf'}, 'distance': 0.43276262283325195, 'rerank_score': 0.0003362302959430963}, {'id': 'Neural-Network(Basics).pdf_chunk26', 'text': 'Optimization (cont’d)\n\nFind w which minimizes the chosen\nerror function E(w)\n\n● wA : a local minimum\n● wB : a global minimum\n● At point wC local gradient is given\n by vector ΔE(w)\n● It points in direction of greatest\n rate of increase of E(w)\n● Negative gradient points to rate\n of greatest decrease 27', 'metadata': {'source': 'Neural-Network(Basics).pdf'}, 'distance': 0.433167040348053, 'rerank_score': 0.0003239178622607142}, {'id': 'Neural-Network(Basics).pdf_chunk37', 'text': 'Backpropagation Algorithm (cont’d)w\n● Backpropagation – for wij jk\n outermost layer\n ○ outermost layer parameters\n directly affect the value of the\n error function.\n ○ only one term of the E\n summation will have a\n non-zero derivative: the one Input (i) Hidden (j) Output (k)\n associated with the particular\n weight we are considering.\n\n 38', 'metadata': {'source': 'Neural-Network(Basics).pdf'}, 'distance': 0.41671454906463623, 'rerank_score': 0.00028604952967725694}, {'id': 'Neural-Network(Basics).pdf_chunk39', 'text': 'Backpropagation Algorithm (cont’d)\n● Backpropagation – for wij w\n jk\n outermost layer\n\n Input (i) Output (k)\n Hidden (j)\n For sigmoid activation function\n\n 40', 'metadata': {'source': 'Neural-Network(Basics).pdf'}, 'distance': 0.4423246383666992, 'rerank_score': 0.00017232770915143192}]

In [15]:
for key, value in chunks_data[0].items() :
    if key == "text" :
        print(value)

Activation Functions (cont’d)

● Sigmoid

 ○ continuously differentiable
 ○ ranges from 0-1
 ○ not symmetric around the origin
● Tanh
 ○
 ○
 ○ scaled version of the sigmoid
 ○ symmetric around the origin
 ○ vanishing gradient

 17


In [16]:
for chunk in chunks_data :
    print(chunk['text'])

Activation Functions (cont’d)

● Sigmoid

 ○ continuously differentiable
 ○ ranges from 0-1
 ○ not symmetric around the origin
● Tanh
 ○
 ○
 ○ scaled version of the sigmoid
 ○ symmetric around the origin
 ○ vanishing gradient

 17
Optimization (cont’d)

Find w which minimizes the chosen
error function E(w)

● wA : a local minimum
● wB : a global minimum
● At point wC local gradient is given
 by vector ΔE(w)
● It points in direction of greatest
 rate of increase of E(w)
● Negative gradient points to rate
 of greatest decrease 28
Optimization (cont’d)

Find w which minimizes the chosen
error function E(w)

● wA : a local minimum
● wB : a global minimum
● At point wC local gradient is given
 by vector ΔE(w)
● It points in direction of greatest
 rate of increase of E(w)
● Negative gradient points to rate
 of greatest decrease 27
Backpropagation Algorithm (cont’d)w
● Backpropagation – for wij jk
 outermost layer
 ○ outermost layer parameters
 directly affect the value of the
 error function

In [1]:
context_chunks = [
    {
        "chunk_id": 1,
        "text": """
AI-Powered Sentiment Intelligence – SentimentIQ

Problem Statement:
To design and develop an end-to-end automated sentiment analysis system that processes
user-uploaded textual datasets, trains multiple machine learning classifiers,
evaluates their performance, visualizes key sentiment indicators,
and generates a comprehensive analytical report.
"""
    },
    {
        "chunk_id": 2,
        "text": """
Abstract:
SentimentIQ is a complete end-to-end sentiment analysis system that enables users
to perform automated sentiment analysis on textual data without requiring
deep machine learning expertise.
"""
    },
    {
        "chunk_id": 3,
        "text": """
Model Performance

Logistic Regression Accuracy : 0.889
Naive Bayes Accuracy : 0.852
SVM Accuracy : 0.896

Word Clouds:
Positive : great, excellent, love
Negative : bad, worst, boring
"""
    },
    {
        "chunk_id": 4,
        "text": """
Project Chapters

Introduction
Hardware Requirements
Software Requirements
Data Dictionary
Project Task Set
Test Cases
Project Budget
"""
    },
    {
        "chunk_id": 5,
        "text": """
Project Plan

Project Schedule
Timeline
System Design
Use Case Diagram
Data Model
Data Flow Diagram
System Architecture
Class Diagram
"""
    }
]

In [2]:
context = "\n\n".join(
    [
        f"CONTEXT CHUNK {chunk['chunk_id']}\n{chunk['text']}"
        for chunk in context_chunks
    ]
)

print(context)

CONTEXT CHUNK 1

AI-Powered Sentiment Intelligence – SentimentIQ

Problem Statement:
To design and develop an end-to-end automated sentiment analysis system that processes
user-uploaded textual datasets, trains multiple machine learning classifiers,
evaluates their performance, visualizes key sentiment indicators,
and generates a comprehensive analytical report.


CONTEXT CHUNK 2

Abstract:
SentimentIQ is a complete end-to-end sentiment analysis system that enables users
to perform automated sentiment analysis on textual data without requiring
deep machine learning expertise.


CONTEXT CHUNK 3

Model Performance

Logistic Regression Accuracy : 0.889
Naive Bayes Accuracy : 0.852
SVM Accuracy : 0.896

Word Clouds:
Positive : great, excellent, love
Negative : bad, worst, boring


CONTEXT CHUNK 4

Project Chapters

Introduction
Hardware Requirements
Software Requirements
Data Dictionary
Project Task Set
Test Cases
Project Budget


CONTEXT CHUNK 5

Project Plan

Project Schedule
Timeline
Sy

In [3]:
query = "Explain the SentimentIQ project in detail."

prompt = f"""
You are an expert technical assistant.

Use ONLY the provided context to answer the user's question.

If information is not present in the context,
reply with "The provided documents do not contain this information."

-------------------------
CONTEXT

{context}

-------------------------

QUESTION

{query}

Provide:

1. Project overview
2. Problem statement
3. Objectives
4. Architecture
5. Model performance
6. Conclusion

Generate a detailed answer with headings and bullet points.
"""

In [4]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",          # or qwen2.5:3b
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=1024,
    request_timeout=300,
)

response = llm.invoke(prompt)

print(response.content)

d:\AI-Restart-2026\Complete GEN AI Journey - Notes\DataIngestion_Pipeline\DataIngestionDemo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here’s a detailed explanation of the SentimentIQ project based on the provided context:

**1. Project Overview**

SentimentIQ is a complete end-to-end sentiment analysis system designed for automated analysis of textual data. It allows users to perform sentiment analysis without needing deep expertise in machine learning.

**2. Problem Statement**

The primary goal of the SentimentIQ project is to develop an automated sentiment analysis system that can:
*   Process user-uploaded textual datasets.
*   Train multiple machine learning classifiers.
*   Evaluate their performance.
*   Visualize key sentiment indicators.
*   Generate a comprehensive analytical report.

**3. Objectives**

The objectives of the SentimentIQ project are to create a fully functional system capable of:
*   Automated sentiment analysis of textual data.
*   Providing insights without requiring deep machine learning knowledge from the user.

**4. Architecture**

While specific architectural diagrams aren’t provided, 